PROJECT TITLE: SpendDNA: BANK TRANSACTION ANALYSIS
NAME: SHANIKA GRACES
BATCH:DATA SCIENCE/DATA ANALYTICS JULY 2026
DATE: 08/08/2026

FEATURE 1-THE TRANSACTION PARSER

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("sample_data/rahul_transactions.csv")
print(df.shape)

(1328, 8)


In [ ]:
df["Date"] = pd.to_datetime(
    df["Date"],dayfirst=True,errors="coerce")
print(df["Date"].dtype)

datetime64[ns]


In [ ]:
df["Amount"] = (
    df["Amount"]
    .astype(str)
    .str.replace("₹", "", regex=False)
    .str.replace("Rs.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip())
df["Amount"] = pd.to_numeric(
    df["Amount"],
    errors="coerce")
print(df["Amount"].dtype)

float64


In [ ]:
df["Type"] = df["Type"].str.lower().str.strip()
df["Type"] = df["Type"].replace({
    "dr": "debit",
    "cr": "credit"})
df["Type"].unique()

array(['debit', 'credit'], dtype=object)

In [ ]:
error_dates = df["Date"].isna().sum()
error_amounts = df["Amount"].isna().sum()
print("Unparseable dates:", error_dates)
print("Unparseable amounts:", error_amounts)

Unparseable dates: 0
Unparseable amounts: 0


In [ ]:
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)
df_clean = df.drop_duplicates().copy()
print(df_clean.shape)
print(df_clean.dtypes)

Duplicate rows: 18
(1310, 8)
Date           datetime64[ns]
Time                   object
Description            object
Type                   object
Amount                float64
Balance               float64
Mode                   object
Ref                    object
dtype: object


FEATURE 2-VENDOR EXTRACTOR

In [ ]:
vendor_keywords = {
    "Swiggy": ["SWIGGY", "BUNDL", "INSTAMART"],
    "Zomato": ["ZOMATO"],
    "Zepto": ["ZEPTO", "KIRANAKART"],
    "Blinkit": ["BLINKIT", "GROFERS"],
    "Amazon": ["AMAZON", "AMZN"],
    "Flipkart": ["FLIPKART", "FLIPKART INTERNET", "FKART INTRNET"],
    "Myntra": ["MYNTRA"],
    "Nykaa": ["NYKAA"],
    "BigBasket": ["BIGBASKET", "INNOVATIVE RETAIL"],
    "DMart": ["DMART", "AVENUE SUPERMARTS"],
    "Uber": ["UBER"],
    "Ola": ["OLA", "ANI TECHNOLOGIES"],
    "Rapido": ["RAPIDO", "ROPPEN TRANSPORTATION"],
    "BMTC": ["BMTC"],
    "Starbucks": ["STARBUCKS"],
    "Third Wave Coffee": ["THIRD WAVE", "THIRDWAVE", "TWC INDIA"],
    "Cafe Coffee Day": ["COFFEE DAY", "CCD", "CAFE COFFEE DAY"],
    "Empire Restaurant": ["EMPIRE RESTAURANT"],
    "Meghana Foods": ["MEGHANA FOODS"],
    "Restaurants": ["BANGALORE RESTAURANT"],
    "Indian Oil": ["INDIAN OIL"],
    "HP Petrol": ["HP PETROL"],
    "Dineout": ["DINEOUT"],
    "BMS": ["BMS MOVIE"],
    "Disney+ Hotstar": ["DISNEY HOTSTAR"],
    "Zerodha": ["ZERODHA"],
    "BESCOM": ["BESCOM", "BANGALORE ELEC SUPPLY"],
    "BWSSB": ["BWSSB"],
    "Reliance Jio": ["RELIANCE JIO"],
    "Airtel": ["AIRTEL"],
    "Techcrush Labs": ["TECHCRUSH LABS"],
    "Truffles": ["TRUFFLES"],
    "Vodafone Idea": ["VI POSTPAID", "VODAFONE IDEA"],
    "Netflix": ["NETFLIX"],
    "Nykaa": ["NYKAA", "FSN E-COMMERCE"],
    "Spotify": ["SPOTIFY"],
    "Disney+ Hotstar": ["DISNEY HOTSTAR", "STAR INDIA"],
    "BPCL": ["BPCL"],
    "BMS": ["BMS MOVIE", "BIGTREE ENTERTAINMENT"],
    "Groww": ["GROWW", "NEXTBILLION"],
    "Reliance Jio": ["RELIANCE JIO", "JIOFIBER"]
}

In [ ]:
def extract_vendor(description):
    description = str(description).upper()
    if "ATM-WDL" in description or "ATM WDL" in description:
        return "Cash Withdrawal"
    if "IMPS-RENT-LANDLORD" in description:
        return "Rent"
    for vendor, keywords in vendor_keywords.items():
        for keyword in keywords:
            if keyword in description:
                return vendor
    if description.startswith("UPI-"):
        return "P2P Transfer"
    return "Other"

In [ ]:
df_clean["vendor_clean"] = df_clean["Description"].apply(extract_vendor)
print("Unique vendors:", df_clean["vendor_clean"].nunique())
print(df_clean["vendor_clean"].value_counts().head(10))

Unique vendors: 40
vendor_clean
Swiggy          243
Zomato          121
Ola              87
Amazon           86
Zepto            71
Uber             71
Blinkit          55
Rapido           55
P2P Transfer     53
Flipkart         47
Name: count, dtype: int64


FEATURE 3:CATEGORY MAPPING

In [ ]:
category_map = {
    "Swiggy":"Food Delivery", "Zomato":"Food Delivery",

    "Zepto":"Quick Commerce", "Blinkit":"Quick Commerce",

    "Amazon":"Ecommerce", "Flipkart":"Ecommerce", "Myntra":"Ecommerce", "Nykaa":"Ecommerce",

    "BigBasket":"Groceries", "DMart":"Groceries",

    "Uber":"Transport", "Ola":"Transport", "Rapido":"Transport", "BMTC":"Transport",

    "Starbucks":"Cafe", "Third Wave Coffee":"Cafe", "Cafe Coffee Day":"Cafe",

    "Truffles":"Restaurants", "Empire Restaurant":"Restaurants", "Meghana Foods":"Restaurants", "Dineout":"Restaurants","Restaurants": "Restaurants",

    "Netflix":"Subscriptions", "Spotify":"Subscriptions", "Disney+ Hotstar":"Subscriptions",

    "BESCOM":"Utilities", "BWSSB":"Utilities", "Reliance Jio":"Utilities",

    "Airtel":"Utilities", "Vodafone Idea":"Utilities", "Rent":"Utilities",

    "Groww":"Investments", "Zerodha":"Investments",

    "Indian Oil":"Fuel", "HP Petrol":"Fuel", "BPCL":"Fuel",

    "BMS":"Entertainment",

    "P2P Transfer":"Personal Transfer", "Cash Withdrawal":"Cash Withdrawal"
}

In [ ]:
df_clean["category"] = df_clean["vendor_clean"].map(category_map).fillna("Other")
print(df_clean["category"].value_counts())

category
Food Delivery        364
Transport            250
Ecommerce            172
Quick Commerce       126
Cafe                  99
Restaurants           56
Personal Transfer     53
Utilities             45
Groceries             41
Subscriptions         27
Investments           23
Fuel                  22
Cash Withdrawal       17
Entertainment          9
Other                  6
Name: count, dtype: int64


FEATURE 4:SPENDING OVERVIEW

In [ ]:
credits=df_clean.loc[df_clean.Type=="credit","Amount"].sum()
debits=df_clean.loc[df_clean.Type=="debit","Amount"].sum()

print("Credits:",credits)
print("Debits:",debits)
print("Net:",credits-debits)
print("Savings Rate:",f"{(credits-debits)/credits*100:.1f}%")
print("Top Categories:",df_clean.groupby("category").Amount.sum().nlargest(5))
print("Top Vendors:",df_clean.groupby("vendor_clean").Amount.sum().nlargest(5))
print("Transactions:",len(df_clean))

Credits: 509774.0
Debits: 1678901.0
Net: -1169127.0
Savings Rate: -229.3%
Top Categories: category
Ecommerce        603877.0
Other            509774.0
Investments      248160.0
Food Delivery    159667.0
Utilities        147924.0
Name: Amount, dtype: float64
Top Vendors: vendor_clean
Techcrush Labs    509774.0
Amazon            328530.0
Zerodha           210000.0
Flipkart          177510.0
Rent              108000.0
Name: Amount, dtype: float64
Transactions: 1310


In [ ]:
print(df_clean.groupby("category").Amount.sum().nlargest(5))

category
Ecommerce        603877.0
Other            509774.0
Investments      248160.0
Food Delivery    159667.0
Utilities        147924.0
Name: Amount, dtype: float64


In [ ]:
print(df_clean.groupby("vendor_clean").Amount.sum().nlargest(5))

vendor_clean
Techcrush Labs    509774.0
Amazon            328530.0
Zerodha           210000.0
Flipkart          177510.0
Rent              108000.0
Name: Amount, dtype: float64


In [ ]:
df_clean["hour"]=pd.to_datetime(df_clean.Time,format="%H:%M",errors="coerce").dt.hour

In [ ]:
food=df_clean[df_clean.category=="Food Delivery"]
late=food[food.hour.isin([21,22,23,0,1])]
print("Late-night Food Delivery:",len(late)/len(food)*100,"%")

Late-night Food Delivery: 20.604395604395602 %


FEATURE 5-MONTHLY TREND ANALYSIS

In [ ]:
df_clean['month'] = df_clean['Date'].dt.month
month_pivot = df_clean.pivot_table(
    values="Amount",
    index="category",
    columns="month",
    aggfunc="sum"
).fillna(0)

print(month_pivot)

month                    1        2         3        4        5         6
category                                                                 
Cafe                3690.0   4273.0    5448.0   6564.0   5668.0    5802.0
Cash Withdrawal     2000.0   5000.0    8000.0   5500.0   8000.0   17000.0
Ecommerce          98623.0  94011.0  108215.0  69219.0  95776.0  138033.0
Entertainment        311.0    474.0    1856.0   1046.0      0.0    1914.0
Food Delivery      23692.0  26458.0   25918.0  30092.0  26598.0   26909.0
Fuel               23058.0   2079.0   21035.0  18013.0   7188.0    2882.0
Groceries          17649.0   8571.0    6289.0  11748.0   9718.0    5432.0
Investments        38432.0  15000.0   68644.0  54126.0  48628.0   23330.0
Other              84728.0  84724.0   84701.0  84736.0  85393.0   85492.0
Personal Transfer  16326.0   7180.0   15379.0   5738.0   9209.0   11902.0
Quick Commerce      9995.0  12459.0   12911.0   9534.0  10757.0    9398.0
Restaurants        16320.0  18877.0   

In [ ]:
growth = (month_pivot[6] - month_pivot[1]) / month_pivot[1] * 100

print("Biggest Growth:", growth.idxmax(), f"{growth.max():.1f}%")
print("Biggest Decline:", growth.idxmin(), f"{growth.min():.1f}%")

Biggest Growth: Cash Withdrawal 750.0%
Biggest Decline: Fuel -87.5%


FEATURE 6-TIME-OF-DAY PATTERNS

In [ ]:
df_clean["hour"] = df_clean["Time"].str[:2].astype(int)
time_matrix = df_clean[df_clean["Type"] == "debit"].pivot_table(
    values="Amount",
    index="category",
    columns="hour",
    aggfunc="sum",
    fill_value=0
)
print("Spending by Category and Hour:")
print(time_matrix)

Spending by Category and Hour:
hour                    0        1        2        3        4        5   \
category                                                                  
Cafe                 872.0    549.0      0.0    316.0      0.0      0.0   
Cash Withdrawal        0.0      0.0      0.0      0.0      0.0      0.0   
Ecommerce          18650.0   9801.0  10246.0  14865.0  12018.0  17258.0   
Entertainment          0.0    620.0      0.0      0.0      0.0      0.0   
Food Delivery       1515.0   3017.0   1374.0   2135.0   2618.0   3488.0   
Fuel                2675.0      0.0      0.0    676.0   2010.0   2105.0   
Groceries           3895.0   1536.0   2352.0  10064.0   1169.0      0.0   
Investments         4883.0  15000.0      0.0      0.0  18834.0   4496.0   
Personal Transfer    562.0   2048.0   2512.0    393.0    654.0      0.0   
Quick Commerce       859.0   1402.0    697.0    732.0    768.0   1448.0   
Restaurants         1411.0    960.0      0.0   1714.0      0.0   2126

In [ ]:
food = df_clean[df_clean.category == "Food Delivery"]
late = food[food.hour.isin([21,22,23,0,1])]
print("Late-night Food Delivery:",
      round(len(late)/len(food)*100, 1), "%")

Late-night Food Delivery: 20.6 %


In [ ]:
cafe = df_clean[df_clean.category == "Cafe"]
morning = cafe[cafe.hour.isin([8,9,10,11])]
print("Morning Cafe:",
      round(len(morning)/len(cafe)*100, 1), "%")

Morning Cafe: 35.4 %


FEATURE 7-ANAMOLY DETECTION

In [ ]:
df_clean["z_score"] = df_clean.groupby("category")["Amount"].transform(
    lambda x: (x-x.mean())/x.std()
)

In [ ]:
df_clean["anomaly"] = df_clean["z_score"] > 2
print("Anomalies:", df_clean["anomaly"].sum())

Anomalies: 38


In [ ]:
print(df_clean[df_clean.anomaly]
      .sort_values("z_score", ascending=False)
      [["Date","vendor_clean","category","Amount","z_score"]]
      .head(5))

           Date  vendor_clean           category   Amount   z_score
148  2024-01-22  P2P Transfer  Personal Transfer   7264.0  5.667589
1298 2024-06-26        Amazon          Ecommerce  22008.0  4.090349
269  2024-02-07        Amazon          Ecommerce  21986.0  4.085484
475  2024-03-05        Amazon          Ecommerce  19917.0  3.627956
414  2024-02-26   Restaurants        Restaurants   8383.0  3.390234


FEATURE 8-SPENDING ARCHETYPE DETECTION

In [ ]:
debits = df_clean[df_clean["Type"] == "debit"]
total_spend = debits["Amount"].sum()

pct = (
    debits.groupby("category")["Amount"].sum()
    / total_spend * 100
)
archetypes = {
    "Foodie": pct.get("Food Delivery", 0)
             + pct.get("Restaurants", 0)
             + pct.get("Cafe", 0) > 25,

    "Quick Commerce Junkie": pct.get("Quick Commerce", 0) > 15,

    "Shopaholic": pct.get("Ecommerce", 0) > 15,

    "Investor": pct.get("Investments", 0) > 15,

    "Cab Commuter": pct.get("Transport", 0) > 10,

    "Subscription Lover":
        debits[debits["category"] == "Subscriptions"]["vendor_clean"].nunique() >= 5,

    "YOLO Spender":
        ((credits - total_spend) / credits) < 0.10,

    "Disciplined Saver":
        ((credits - total_spend) / credits) > 0.40
}
print("Rahul's Spending Archetypes:\n")
for name, matched in archetypes.items():
    if matched:
        print(name)

Rahul's Spending Archetypes:

Shopaholic
YOLO Spender
